# Bengali NLP Safety Challenge 2026 — Hybrid Ensemble (v3)

**Metric target:** macro-F1 (leaderboard is F1-based). Rare attribute labels dominate the score, so this notebook is built to maximize them.

**Hybrid ensemble**
1. **Transformer + RNN hybrid:** BanglaBERT-large -> BiLSTM -> attention + max pooling head
2. **Classical lexical model:** TF-IDF (word + char n-grams) -> per-label Logistic Regression
3. **Blend** the two (weight auto-tuned on OOF) -> per-label **F1 threshold tuning** -> `is_toxic` **gating**

**Imbalance handling:** Focal loss + per-label `pos_weight` + light EDA augmentation that oversamples rare-positive rows.

> Requires **GPU ON** + **Internet ON**.


In [ ]:
# ============================================================
# 0. Install dependencies (Internet must be ON)
# ============================================================
import sys, subprocess
def pip(*a): subprocess.run([sys.executable,"-m","pip","install","-q",*a], check=False)
pip("iterative-stratification==0.1.7")
pip("git+https://github.com/csebuetnlp/normalizer")
print("deps done")


In [ ]:
# ============================================================
# 1. Imports & config
# ============================================================
import os, re, gc, random, warnings, glob
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score
warnings.filterwarnings("ignore")
from transformers import (AutoTokenizer, AutoModel, AutoConfig,
                          get_linear_schedule_with_warmup, logging as hf_logging)
hf_logging.set_verbosity_error()

class CFG:
    # Backbone: banglabert_large is strongest; fall back to base if you hit OOM.
    model_name    = "csebuetnlp/banglabert_large"
    fallback_name = "csebuetnlp/banglabert"
    max_len       = 160
    n_folds       = 5
    epochs        = 4
    batch_size    = 8
    grad_accum    = 2          # effective batch = 16
    eval_bs       = 32
    lr            = 1.5e-5
    weight_decay  = 0.01
    warmup_ratio  = 0.06
    max_grad_norm = 1.0
    dropout       = 0.3
    lstm_hidden   = 256
    focal_gamma   = 2.0
    pos_weight_clip = 20.0
    seed          = 42
    use_fp16      = True
    AUGMENT       = True       # EDA oversampling of rare-positive rows
    aug_copies    = 2          # copies per rare-positive training row
    GATE          = True       # zero out attributes where is_toxic predicted 0
    METRIC        = "f1"       # threshold-tuning objective ("f1" or "accuracy")

LABELS = ["is_toxic","implicit_toxicity","sarcasm","contains_profanity",
          "threatening","identity_attack","sexual_content"]
RARE   = ["implicit_toxicity","sarcasm","threatening","identity_attack","sexual_content"]
N_LABELS = len(LABELS)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(CFG.seed)


In [ ]:
# ============================================================
# 2. Load data (recursive auto-detect)
# ============================================================
def find_data_dir():
    # Prefer the OFFICIAL competition mount (its test.csv IDs are what Kaggle scores)
    for c in ["/kaggle/input/competitions/bengali-nlp-safety-challenge-2026",
              "/kaggle/input/bengali-nlp-safety-challenge-2026"]:
        if os.path.exists(os.path.join(c,"train.csv")): return c
    hits = glob.glob("/kaggle/input/**/train.csv", recursive=True)
    if hits: return os.path.dirname(sorted(hits, key=len)[0])
    for c in ["./data","."]:
        if os.path.exists(os.path.join(c,"train.csv")): return c
    print("Contents of /kaggle/input:")
    for r,_,fs in os.walk("/kaggle/input"):
        for f in fs: print("   ", os.path.join(r,f))
    raise FileNotFoundError("Attach the competition dataset as input.")

DATA_DIR = find_data_dir(); print("Data dir:", DATA_DIR)
train_df = pd.read_csv(os.path.join(DATA_DIR,"train.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR,"test.csv"))
try: val_df = pd.read_csv(os.path.join(DATA_DIR,"validation.csv"))
except FileNotFoundError: val_df = None
try: sample_sub = pd.read_csv(os.path.join(DATA_DIR,"sample_submission.csv"))
except FileNotFoundError: sample_sub = None

def pick_col(df, prefer, kw):
    for p in prefer:
        if p in df.columns: return p
    for c in df.columns:
        if any(k in c.lower() for k in kw): return c
    obj=[c for c in df.columns if df[c].dtype==object]
    return max(obj, key=lambda c: df[c].astype(str).str.len().mean())
TEXT_COL = pick_col(train_df, ["text"], ["text","comment","sentence","content"])
ID_COL   = pick_col(test_df,  ["ID","id","Id"], ["id"])
print("TEXT_COL =", TEXT_COL, "| ID_COL =", ID_COL)

if val_df is not None and set(LABELS).issubset(val_df.columns):
    full_df = pd.concat([train_df, val_df], ignore_index=True); print("train+validation")
else:
    full_df = train_df.copy(); print("train only")
full_df = full_df.dropna(subset=[TEXT_COL]).reset_index(drop=True)
for l in LABELS: full_df[l] = full_df[l].astype(int)
print("Full:", full_df.shape, "| Test:", test_df.shape)
print(full_df[LABELS].mean().round(3).to_dict())


In [ ]:
# ============================================================
# 3. Cleaning / normalization + EDA augmentation
# ============================================================
try:
    from normalizer import normalize as bn_normalize; _HAS_NORM=True; print("normalizer ON")
except Exception as e:
    _HAS_NORM=False; print("basic cleaning:", e)

URL_RE=re.compile(r"http\S+|www\.\S+"); HTML_RE=re.compile(r"<.*?>")
SPACE_RE=re.compile(r"\s+"); KEEP_RE=re.compile(r"[^\u0980-\u09FF\u0020-\u007E]")
def clean_text(t):
    t=str(t); t=HTML_RE.sub(" ",t); t=URL_RE.sub(" ",t)
    t=t.replace("\u200c"," ").replace("\u200d"," "); t=KEEP_RE.sub(" ",t)
    t=SPACE_RE.sub(" ",t).strip()
    if _HAS_NORM:
        try: t=bn_normalize(t)
        except Exception: pass
    return t if t else "[EMPTY]"
full_df["clean"]=full_df[TEXT_COL].apply(clean_text)
test_df["clean"]=test_df[TEXT_COL].apply(clean_text)

def eda(text, p_del=0.1):
    w=text.split()
    if len(w)<4: return text
    w=[x for x in w if random.random()>p_del] or w
    if len(w)>=2:
        i,j=random.sample(range(len(w)),2); w[i],w[j]=w[j],w[i]
    return " ".join(w)

def augment_indices(sub_df):
    # returns extra (text, label) rows created from rare-positive rows
    ex_txt, ex_lab = [], []
    if not CFG.AUGMENT: return ex_txt, ex_lab
    mask = sub_df[RARE].sum(axis=1) > 0
    rare_rows = sub_df[mask]
    for _ in range(CFG.aug_copies):
        for _, r in rare_rows.iterrows():
            ex_txt.append(eda(r["clean"]))
            ex_lab.append(r[LABELS].values.astype(float))
    return ex_txt, ex_lab
print(full_df[["clean"]].head(2).to_string())


In [ ]:
# ============================================================
# 4. Tokenizer, Dataset, Hybrid model, Focal loss
# ============================================================
try:
    tokenizer = AutoTokenizer.from_pretrained(CFG.model_name); MODEL_NAME=CFG.model_name
except Exception as e:
    print("large tokenizer failed, using base:", e)
    tokenizer = AutoTokenizer.from_pretrained(CFG.fallback_name); MODEL_NAME=CFG.fallback_name
print("Backbone:", MODEL_NAME)

class ToxDS(Dataset):
    def __init__(self, texts, labels=None):
        self.texts=list(texts); self.labels=labels
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc=tokenizer(self.texts[i], truncation=True, max_length=CFG.max_len,
                      padding="max_length", return_tensors="pt")
        item={k:v.squeeze(0) for k,v in enc.items()}
        if self.labels is not None:
            item["labels"]=torch.tensor(self.labels[i], dtype=torch.float)
        return item

class HybridModel(nn.Module):
    def __init__(self, model_name, n_labels):
        super().__init__()
        self.config=AutoConfig.from_pretrained(model_name)
        self.backbone=AutoModel.from_pretrained(model_name)
        h=self.config.hidden_size; L=CFG.lstm_hidden
        self.lstm=nn.LSTM(h, L, batch_first=True, bidirectional=True)
        self.attn=nn.Linear(2*L, 1)
        self.drop=nn.Dropout(CFG.dropout)
        self.head=nn.Sequential(nn.Linear(4*L, 2*L), nn.GELU(),
                                nn.Dropout(CFG.dropout), nn.Linear(2*L, n_labels))
    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kw=dict(input_ids=input_ids, attention_mask=attention_mask)
        if token_type_ids is not None: kw["token_type_ids"]=token_type_ids
        seq=self.backbone(**kw).last_hidden_state          # B,T,H
        lo,_=self.lstm(seq)                                 # B,T,2L
        m=attention_mask.unsqueeze(-1)
        sc=self.attn(lo).masked_fill(m==0, torch.finfo(lo.dtype).min)
        w=torch.softmax(sc, dim=1)
        attn_pool=(w*lo).sum(1)                             # B,2L
        max_pool=lo.masked_fill(m==0, torch.finfo(lo.dtype).min).max(1).values   # B,2L
        feat=torch.cat([attn_pool, max_pool], -1)           # B,4L
        return self.head(self.drop(feat))

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__(); self.gamma=gamma; self.pos_weight=pos_weight
    def forward(self, logits, targets):
        bce=F.binary_cross_entropy_with_logits(logits, targets, reduction="none",
                                               pos_weight=self.pos_weight)
        p=torch.sigmoid(logits); pt=p*targets+(1-p)*(1-targets)
        return (((1-pt)**self.gamma)*bce).mean()
print("model ready")


In [ ]:
# ============================================================
# 5. Transformer training (one fold) with focal loss, grad-accum, fp16
# ============================================================
def pos_weight_from(y):
    pos=y.sum(0); neg=len(y)-pos
    return np.clip(neg/np.clip(pos,1,None), 1.0, CFG.pos_weight_clip)

@torch.no_grad()
def predict_probs(model, loader):
    model.eval(); out=[]
    for b in loader:
        ids=b["input_ids"].to(DEVICE); mask=b["attention_mask"].to(DEVICE)
        tti=b.get("token_type_ids"); tti=tti.to(DEVICE) if tti is not None else None
        with torch.cuda.amp.autocast(enabled=CFG.use_fp16):
            logit=model(ids,mask,tti)
        out.append(torch.sigmoid(logit).float().cpu().numpy())
    return np.concatenate(out,0)

def train_transformer_fold(tr_texts, tr_y, va_texts, va_y, test_texts, fold):
    seed_everything(CFG.seed+fold)
    model=HybridModel(MODEL_NAME, N_LABELS).to(DEVICE)
    tr_loader=DataLoader(ToxDS(tr_texts,tr_y), batch_size=CFG.batch_size, shuffle=True,
                         num_workers=0, pin_memory=True)
    va_loader=DataLoader(ToxDS(va_texts,va_y), batch_size=CFG.eval_bs, shuffle=False, num_workers=0)
    te_loader=DataLoader(ToxDS(test_texts),    batch_size=CFG.eval_bs, shuffle=False, num_workers=0)
    pw=torch.tensor(pos_weight_from(tr_y), dtype=torch.float, device=DEVICE)
    crit=FocalLoss(CFG.focal_gamma, pos_weight=pw)
    nd=["bias","LayerNorm.weight"]
    params=[{"params":[p for n,p in model.named_parameters() if not any(x in n for x in nd)],"weight_decay":CFG.weight_decay},
            {"params":[p for n,p in model.named_parameters() if any(x in n for x in nd)],"weight_decay":0.0}]
    opt=torch.optim.AdamW(params, lr=CFG.lr)
    steps=(len(tr_loader)//CFG.grad_accum)*CFG.epochs
    sch=get_linear_schedule_with_warmup(opt, int(steps*CFG.warmup_ratio), steps)
    scaler=torch.cuda.amp.GradScaler(enabled=CFG.use_fp16)
    best_f1,best_val,best_test=-1,None,None
    for ep in range(CFG.epochs):
        model.train(); run=0.0; opt.zero_grad()
        for st,b in enumerate(tr_loader):
            ids=b["input_ids"].to(DEVICE); mask=b["attention_mask"].to(DEVICE)
            tti=b.get("token_type_ids"); tti=tti.to(DEVICE) if tti is not None else None
            y=b["labels"].to(DEVICE)
            with torch.cuda.amp.autocast(enabled=CFG.use_fp16):
                loss=crit(model(ids,mask,tti), y)/CFG.grad_accum
            scaler.scale(loss).backward(); run+=loss.item()*CFG.grad_accum
            if (st+1)%CFG.grad_accum==0:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),CFG.max_grad_norm)
                scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad()
        vp=predict_probs(model,va_loader)
        macro=f1_score(va_y,(vp>=0.5).astype(int),average="macro",zero_division=0)
        print(f"  fold{fold} ep{ep+1}/{CFG.epochs} loss={run/len(tr_loader):.4f} valMacroF1(0.5)={macro:.4f}")
        if macro>=best_f1:
            best_f1=macro; best_val=vp; best_test=predict_probs(model,te_loader)
    del model; gc.collect(); torch.cuda.empty_cache()
    return best_val, best_test


In [ ]:
# ============================================================
# 6. Classical TF-IDF (word+char) -> per-label Logistic Regression
# ============================================================
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def train_tfidf_fold(tr_texts, tr_y, va_texts, test_texts):
    wv=TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=2, max_features=60000, sublinear_tf=True)
    cv=TfidfVectorizer(analyzer="char_wb", ngram_range=(2,5), min_df=2, max_features=60000, sublinear_tf=True)
    Xtr=hstack([wv.fit_transform(tr_texts), cv.fit_transform(tr_texts)]).tocsr()
    Xva=hstack([wv.transform(va_texts),   cv.transform(va_texts)]).tocsr()
    Xte=hstack([wv.transform(test_texts), cv.transform(test_texts)]).tocsr()
    va_p=np.zeros((Xva.shape[0],N_LABELS)); te_p=np.zeros((Xte.shape[0],N_LABELS))
    for i in range(N_LABELS):
        yi=tr_y[:,i].astype(int)
        if yi.sum()==0:
            continue
        clf=LogisticRegression(C=3.0, class_weight="balanced", max_iter=1000)
        clf.fit(Xtr, yi)
        va_p[:,i]=clf.predict_proba(Xva)[:,1]
        te_p[:,i]=clf.predict_proba(Xte)[:,1]
    return va_p, te_p
print("tfidf fns ready")


In [ ]:
# ============================================================
# 7. Cross-validated training (creates oof_tf, oof_clf, test_tf, test_clf)
# ============================================================
try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
    splitter=MultilabelStratifiedKFold(n_splits=CFG.n_folds,shuffle=True,random_state=CFG.seed)             .split(full_df["clean"].values, full_df[LABELS].values)
    print("MultilabelStratifiedKFold")
except Exception as e:
    from sklearn.model_selection import StratifiedKFold
    splitter=StratifiedKFold(n_splits=CFG.n_folds,shuffle=True,random_state=CFG.seed)             .split(full_df["clean"].values, full_df["is_toxic"].values)
    print("StratifiedKFold fallback:", e)

X=full_df["clean"].values
Y=full_df[LABELS].values.astype(float)
test_texts=test_df["clean"].values
oof_tf=np.zeros((len(full_df),N_LABELS));  oof_clf=np.zeros((len(full_df),N_LABELS))
test_tf=np.zeros((len(test_df),N_LABELS)); test_clf=np.zeros((len(test_df),N_LABELS))

for fold,(tr_idx,va_idx) in enumerate(splitter):
    print(f"===== Fold {fold+1}/{CFG.n_folds} =====")
    sub_tr=full_df.iloc[tr_idx]
    tr_texts=list(X[tr_idx]); tr_y=Y[tr_idx].copy()
    ex_txt,ex_lab=augment_indices(sub_tr)
    if ex_txt:
        tr_texts=tr_texts+ex_txt; tr_y=np.vstack([tr_y, np.array(ex_lab)])
        print(f"  augmented +{len(ex_txt)} rare-positive rows -> {len(tr_texts)} train rows")
    # transformer
    vtf,ttf=train_transformer_fold(np.array(tr_texts,dtype=object),tr_y,X[va_idx],Y[va_idx],test_texts,fold)
    oof_tf[va_idx]=vtf; test_tf+=ttf/CFG.n_folds
    # tfidf
    vclf,tclf=train_tfidf_fold(list(X[tr_idx]),Y[tr_idx],list(X[va_idx]),list(test_texts))
    oof_clf[va_idx]=vclf; test_clf+=tclf/CFG.n_folds
print("CV training complete.")


In [ ]:
# ============================================================
# 8. Blend + threshold tuning for the EXACT competition metric
#    Combined = (A + B)/2
#      A = is_toxic MACRO-F1 (both classes)
#      B = mean of the 6 attribute POSITIVE-class F1 scores
# ============================================================
def gate_preds(prob, thr, gate=True):
    pred=(prob>=thr).astype(int)
    if gate: pred[pred[:,0]==0, 1:]=0
    return pred

def combined_score(y, pred):
    A=f1_score(y[:,0], pred[:,0], average="macro", zero_division=0)
    B=np.mean([f1_score(y[:,i], pred[:,i], zero_division=0) for i in range(1,N_LABELS)])
    return (A+B)/2.0, A, B

def tune_thresholds_metric(y, prob):
    grid=np.arange(0.05,0.991,0.01); thr=np.full(N_LABELS,0.5)
    # is_toxic -> maximize MACRO-F1 (both classes)
    bt,bs=0.5,-1
    for t in grid:
        s=f1_score(y[:,0],(prob[:,0]>=t).astype(int),average="macro",zero_division=0)
        if s>bs: bs,bt=s,t
    thr[0]=bt
    # attributes -> maximize positive-class F1
    for i in range(1,N_LABELS):
        bt,bs=0.5,-1
        for t in grid:
            s=f1_score(y[:,i],(prob[:,i]>=t).astype(int),zero_division=0)
            if s>bs: bs,bt=s,t
        thr[i]=bt
    return thr

best=(-1,None,None)
for w in np.round(np.arange(0.0,1.001,0.05),2):
    blend=w*oof_tf+(1-w)*oof_clf
    thr=tune_thresholds_metric(Y, blend)
    c,_,_=combined_score(Y, gate_preds(blend,thr,CFG.GATE))
    if c>best[0]: best=(c,w,thr)
best_c,best_w,thr=best

oof_blend=best_w*oof_tf+(1-best_w)*oof_clf
oof_pred=gate_preds(oof_blend, thr, CFG.GATE)
c,A,B=combined_score(Y, oof_pred)
print(f"Best blend weight  w(transformer)={best_w}  (1-w on TF-IDF)")
print("-"*52)
print(f"OOF Subtask A  is_toxic macro-F1 = {A:.4f}")
print(f"OOF Subtask B  mean 6-attr F1    = {B:.4f}")
print(f"OOF COMBINED SCORE               = {c:.4f}   <-- expected leaderboard")
print("-"*52)
print("Per-attribute positive-F1 & threshold:")
for i,l in enumerate(LABELS):
    f=f1_score(Y[:,i],oof_pred[:,i],average=("macro" if i==0 else "binary"),zero_division=0)
    print(f"  {l:<20} thr={thr[i]:.2f}  F1={f:.4f}")


In [ ]:
# ============================================================
# 9. Build submission (match test.csv IDs — the accepted format)
# ============================================================
test_blend=best_w*test_tf+(1-best_w)*test_clf
final=gate_preds(test_blend, thr, CFG.GATE)
sub=pd.DataFrame({ID_COL:test_df[ID_COL].values})
for i,l in enumerate(LABELS): sub[l]=final[:,i]
sub.to_csv("submission.csv", index=False)
print("submission.csv:", sub.shape, "| IDs taken from official test.csv")
print(sub.head())
print("Positive rate per label:")
print(sub[LABELS].mean().round(3))


## Tuning knobs (if you still need more)
- `CFG.epochs` 4->6, `CFG.n_folds` 5->7 (more time, higher score).
- Add a 3rd backbone (e.g. `google/muril-base-cased` or `xlm-roberta-large`) and average its probs into the blend.
- `CFG.aug_copies` 2->3 and lower `p_del` if rare labels overfit.
- If OOM on large: set `CFG.model_name = CFG.fallback_name`, or `CFG.batch_size=4, CFG.grad_accum=4`.
